[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mlatcl/multilogue/blob/main/notebooks/05_multiverse.ipynb)

# Multiverse — which decisions move $\ell$?

### [Neil D. Lawrence](http://inverseprobability.com), University of Cambridge

### 2026-10-17

**Abstract**: A single pipeline path hides researcher degrees of freedom. Following
Bell, Kampman, Dodge & Lawrence (2022), multilogue enumerates a Cartesian
product of analytic choices and asks a consumer-supplied callback for the
headline metric $\ell$ under each configuration. Variance shares say which
decisions move $\ell$. This notebook teaches the pattern with a transparent toy
$\ell$ so the figures stay offline.


In [ ]:
#@title Setup (run first)
# Infrastructure only: install or locate multilogue, prepare figure directory.
# Scientific imports live in later cells.

import sys, subprocess, pathlib

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "matplotlib", "pandas", "numpy", "scikit-learn", "pyyaml", "pot",
        "git+https://github.com/mlatcl/multilogue.git",
    ])
else:
    try:
        import multilogue  # noqa: F401
    except ImportError:
        _here = pathlib.Path.cwd().resolve()
        for _cand in (_here, _here.parent, *_here.parents):
            if (_cand / "pyproject.toml").is_file() and (_cand / "multilogue").is_dir():
                if str(_cand) not in sys.path:
                    sys.path.insert(0, str(_cand))
                break

from multilogue.notebook import setup

FIG = setup(figures_dir="outputs/notebook_figures")
print("Figures →", FIG)


# Introduction

The shared library must not hard-code paper-specific outcomes (format gaps,
technology distinctiveness, …). Consumers define $\ell$; multilogue reports how
much of $\ell$'s variance across the multiverse is associated with each
dimension. Shares are a first-cut one-way decomposition — interactions mean they
need not sum to one.


## The callback contract

For each configuration $c$ in the Cartesian product of dimensions, the consumer
returns a dict containing at least the scalar $\ell(c)$. Multilogue never
inspects *how* $\ell$ was computed — only that it is comparable across the grid.


In [ ]:
from IPython.display import display
import numpy as np
from multilogue.multiverse import (
    enumerate_configurations, run_multiverse, variance_decomposition,
)
from multilogue import viz


In [ ]:
dims = {"k": [4, 5, 6], "threshold": [0.4, 0.5, 0.6], "seed": [0, 1]}
configs = enumerate_configurations(dims)
print(len(configs), "configurations")

def evaluate(cfg):
    # Toy ℓ: depends strongly on threshold, weakly on k, barely on seed.
    ell = 0.5 + 0.4 * cfg["threshold"] + 0.05 * cfg["k"] + 0.01 * cfg["seed"]
    ell += np.random.default_rng(cfg["seed"] + 10 * cfg["k"]).normal(0, 0.01)
    return {"ell": ell}

results = run_multiverse(configs, evaluate)
display(results.head())
shares = variance_decomposition(results, "ell", list(dims))
display(shares)


In [ ]:
viz.variance_share_bars(shares, path=FIG / "05_variance_shares.png")
viz.multiverse_metric_strip(
    results, "ell", colour_by="threshold", path=FIG / "05_metric_strip.png",
)


In this toy, threshold dominates the variance shares — by construction. On real
studies the same plots ask an empirical question: which analytic decisions move
the headline? Notebook 09 wires a *format-gap* $\ell$ through the full Address
stack and reports the same decomposition.


# Outlook

Validation scaffolding — agreement statistics, stratified check sheets, evidence
packs — lives in notebook 06. Soft geometry and document EMD live in notebook 08.
